# ============================================================
# Geospatial mapping pipeline for hyphal network density
# ============================================================
# This script implements a complete workflow to map arbuscular
# mycorrhizal hyphal network density at global  scales. This workflow was largely 
# inspired by https://www.biorxiv.org/content/10.1101/2021.07.07.451145v1
#
# Workflow assumptions and inputs:
# - Geolocated observations of hyphal density are already compiled.
# - Each observation has been sampled against environmental rasters
#   and any project specific predictor variables (e.g. soil core volume).
#
# Main steps:
# 1. Upload the prepared training table to Google Earth Engine.
#    If command line authentication is not configured, upload the
#    table through the Earth Engine Code Editor or API and point
#    the script to that asset.
# 2. Generate custom cross validation folds that respect the study
#    design and spatial structure of the data.
# 3. Train a set of candidate models (for example 30 Random Forest
#    configurations) using these folds.
# 4 Rank models based on cross validated performance and retain
#    the best subset (for example the top 10 models).
# 5. Build an ensemble by drawing bootstrap samples of the training
#    data (100 bootstraps) and predicting across the
#    full environmental raster stack to obtain:
#       - Ensemble mean predictions of hyphal network density.
#       - Uncertainty estimates from the spread of bootstrap
#         predictions.
# 6. Quantify the degree of environmental extrapolation by
#    measuring how far each prediction pixel lies outside the
#    range of predictor values encountered in the training data.
#
# Outputs:
# - Maps of ensemble mean hyphal network density.
# - Maps of modeel prediction uncertainty (SD and COV).
# - Maps describing environmental
#   extrapolation in geographic space.


In [ ]:
# Import the modules of interest
import pandas as pd
import numpy as np
import subprocess
import time
import datetime
import ee
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA
from itertools import combinations
from itertools import repeat

ee.Authenticate()
today = datetime.date.today().strftime("%Y%m%d")



In [ ]:
# Configuration
ee.Initialize()
# Input the name of the username that serves as the home folder for asset storage
usernameFolderString = 'justin618s'
usernameGEE = usernameFolderString
titleOfRawPointCollection = 'hyphal_density_m_cm3'

# Input the Cloud Storage Bucket that will hold the bootstrap collections when uploading them to Earth Engine
# !! This bucket should be pre-created before running this script
bucketOfInterest = 'spun_bucket'

# Input the name of the classification property
classProperty = 'hyphal_density_m_cm3'

#Input an identifier for [Y] - First folder
guild = "Hyphae" 

# Input the name of the project folder inside which all of the assets will be stored
# This folder will be generated automatically below, if it isn't yet present
projectFolder = 'DensityFix/' + guild 

# Input the normal wait time (in seconds) for "wait and break" cells
normalWaitTime = 5

# Input a longer wait time (in seconds) for "wait and break" cells
longWaitTime = 10

# Specify the column names where the latitude and longitude information is stored
latString = 'Pixel_Lat'
longString = 'Pixel_Long'

# Log transform classProperty? Boolean, either True or False
log_transform_classProperty = True

# Ensemble of top 10 models?
ensemble = True

# Spatial leave-one-out cross-validation settings
# skip test points outside training space after removing points in buffer zone? This might reduce extrapolation but overestimate accuracy
loo_cv_wPointRemoval = True

# Define buffer size in meters; use Moran's I or other test to determine SAC range
# Alternatively: specify buffer size as list, to test across multiple buffer sizes
buffer_size = [1000,5000,10000,50000,100000]


In [ ]:
# Import the modules of interest
import pandas as pd
import numpy as np
import subprocess
import time
import datetime
import ee
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA
from itertools import combinations
from itertools import repeat

ee.Authenticate()
today = datetime.date.today().strftime("%Y%m%d")



In [ ]:
# ============================================================
# Covariate image loading and processing
# ============================================================
# This section depends on assembling a full stack of spatial
# predictor layers in Google Earth Engine. These layers represent
# climate, soil chemistry, vegetation structure, land cover, and
# any project specific variables used in the model. The rasters
# are loaded, projected to a common grid, and combined into a
# unified predictor stack that is already sampled at the locations
# of the hyphal density observations.


core_covs = [
  "CGIAR_PET",
  "CHELSA_BIO_Annual_Mean_Temperature",
  "CHELSA_BIO_Annual_Precipitation",
  "CHELSA_BIO_Max_Temperature_of_Warmest_Month",
  "CHELSA_BIO_Precipitation_Seasonality",
  "EarthEnvTexture_CoOfVar_EVI",
  "EarthEnvTexture_Correlation_EVI",
  "EarthEnvTexture_Homogeneity_EVI",
  "EarthEnvTopoMed_AspectCosine",
  "EarthEnvTopoMed_AspectSine",
  "EarthEnvTopoMed_Elevation",
  "EarthEnvTopoMed_Slope",
  "EarthEnvTopoMed_TopoPositionIndex",
  "EsaCci_BurntAreasProbability",
  "GHS_Population_Density",
  "MODIS_NPP",
  "SG_Depth_to_bedrock",
  "SG_SOC_Content_005cm",
  "SG_Soil_pH_H2O_005cm",
  "earthEnvLandcover-class-barren",
  "earthEnvLandcover-class-cultivated-managed-vegetation",
  "earthEnvLandcover-class-evergreen-broadleaf-trees",
  "earthEnvLandcover-class-evergreen-decid-needleleaf-trees",
  "earthEnvLandcover-class-herbaceous-vegetation",
  "earthEnvLandcover-class-mixed-other-trees",
  "earthEnvLandcover-class-regularly-flooded-vegetation",
  "earthEnvLandcover-class-shrubs",
  "harmonized-aboveground-biomass",
  "harmonized-belowground-biomass",
  "isric-soil-proportion-of-sand",
  "HumanModification",
  "myco_veg_cover",
  "nitrogen",
  "phosphorus",
  "sampling_intensity",
  "crops_ESA"
]


## Project-specific (non-raster) variables already in your table
project_vars = [
  "GrowthFormKnown",
  "Growth_Form_Grass",
  "Growth_Form_Herb",
  "Growth_Form_Shrub",
  "Growth_Form_Tree",
  "Season_NorthernHemHarmonized_NA",
  "Season_NorthernHemHarmonized_Spring",
  "Season_NorthernHemHarmonized_Autumn",
  "Season_NorthernHemHarmonized_Summer",
  "Season_NorthernHemHarmonized_Winter",
  "soil_core_volume_cm3",
  "sample_size",
  "soil_depth_depth_cm",
  "cultivated1_wild0_Pironon2023"
]

## Load base image stacks
V1_stack = ee.Image("users/justin618s/V1_composite")
SPUN_stack = ee.Image("users/justin618s/SpunStack_May2023")
reference_projection = V1_stack.projection()

## Load and reproject custom image layers
phosphorus = ee.Image("users/justin618s/phosphorus_total_2022").rename('phosphorus').reproject(reference_projection) 
phosphorus = ee.Image("users/justin618s/GlobalOlsonP_plantavailable").rename('phosphorus').reproject(reference_projection) 



nitrogen = ee.Image("projects/soilgrids-isric/nitrogen_mean").select(0).rename('nitrogen').reproject(reference_projection)
myco_veg_cover = ee.Image("projects/spun-geospatial/assets/mycoFtypeDistributions/MycDistrAM_current").rename('myco_veg_cover').reproject(reference_projection)
sampling_intensity = ee.Image("projects/ee-justin618s/assets/sample_intensity_hyphae_scaled").rename('sampling_intensity').reproject(reference_projection)
human_modification_image = ee.ImageCollection('CSP/HM/GlobalHumanModification').mean().rename('HumanModification').reproject(reference_projection)

## Load crops from ESA WorldCover 2020 (class 40 = cropland)
esa = ee.ImageCollection("ESA/WorldCover/v100").first()
crops_ESA = esa.eq(40).rename("crops_ESA").reproject(reference_projection)
crops_ESA = crops_ESA.unmask(0)
## Load uncultivated grassland from Global Pasture Watch (2022 layer)
pasture_watch = ee.ImageCollection('projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c')
uncult_grass = pasture_watch.filterDate('2020-01-01', '2022-01-01').first().rename("uncultivated_grassland_2010_2022").reproject(reference_projection)

## Reproject stacks
V1_stack = V1_stack.reproject(reference_projection)
SPUN_stack = SPUN_stack.reproject(reference_projection)

## Build composite image with all raster covariates
full_composite = V1_stack.addBands(SPUN_stack).addBands(phosphorus).addBands(nitrogen).addBands(myco_veg_cover).addBands(sampling_intensity).addBands(human_modification_image).addBands(crops_ESA).addBands(uncult_grass)

## Load the composite on which to perform the mapping, and subselect the bands of interest
compositeToClassify = full_composite.select(core_covs)
compositeOfInterest = compositeToClassify

## Scale of composite
scale = full_composite.projection().nominalScale().getInfo()

## Append project-specific vars to covariate list for export
covariateList = core_covs + project_vars

In [ ]:
####################################################################################################################################################################
# Export settings
###################################################################################################################################################################
# Set pyramidingPolicy for exporting purposes
pyramidingPolicy = 'mean'

# Specify CRS to use (of both raw csv and final maps)
CRStoUse = 'EPSG:4326'

# Geometry to use for export
exportingGeometry = ee.Geometry.Polygon([[[-180, 88], [180, 88], [180, -88], [-180, -88]]], None, False);

# Set resolution of final image in arc seconds (30 arc seconds equals to ± 927m at the equator)
export_res = 30

# Convert resolution to degrees
res_deg = export_res/3600

In [ ]:
####################################################################################################################################################################
# Cross validation settings
####################################################################################################################################################################
# Set k for k-fold CV
k = 10

# Make a list of the k-fold CV assignments to use
kList = list(range(1,k+1))

# Set number of trees in RF models
nTrees = 250

# Specify whether to use spatial or random CV
spatialCV = False 

# Input the name of the property that holds the CV fold assignment
cvFoldHeader = 'CV_Fold'

cvFoldString_Spatial = cvFoldHeader + '_Spatial'
cvFoldString_Random = cvFoldHeader + '_Random'
cvFoldString = cvFoldHeader
# Metric to use for sorting k-fold CV hyperparameter tuning (default: R2)
sort_acc_prop = 'Mean_R2' # (either one of 'Mean_R2', 'Mean_MAE', 'Mean_RMSE')

if spatialCV == True:
    sort_acc_prop = sort_acc_prop + '_Spatial'
else:
    sort_acc_prop = sort_acc_prop + '_Random'

# Input the title of the CSV that will hold all of the data that has been given a CV fold assignment
titleOfCSVWithCVAssignments = classProperty +"_training_data"

# Asset ID of uploaded dataset after processing
assetIDForCVAssignedColl = 'users/'+usernameFolderString+'/'+projectFolder+'/'+titleOfCSVWithCVAssignments

# Write the name of a local staging area folder for outputted CSV's
holdingFolder = '/Users/justinstewart/Downloads/'
outputFolder = '/Users/justinstewart/Output'

# Create directory to hold training data
Path(holdingFolder).mkdir(parents=True, exist_ok=True)

In [ ]:
####################################################################################################################################################################
# Bootstrap settings
####################################################################################################################################################################

# Number of bootstrap iterations
bootstrapIterations = 100

# Generate the seeds for bootstrapping
seedsToUseForBootstrapping = list(range(1, bootstrapIterations+1))

# Input the header text that will name the bootstrapped dataset
bootstrapSamples = classProperty+'_bootstrapSamples'

# Write the name of the variable used for stratification
stratificationVariableString = "Resolve_Biome"

# Input the dictionary of values for each of the stratification category levels
# !! This area breakdown determines the proportion of each biome to include in every bootstrap
strataDict = {
    1: 14.900835665820974,
    2: 2.941697660221864,
    3: 0.526059731441294,
    4: 9.56387696566245,
    5: 2.865354077500338,
    6: 11.519674266872787,
    7: 16.26999434439293,
    8: 8.047078485979089,
    9: 0.861212221078014,
    10: 3.623974712557433,
    11: 6.063922959332467,
    12: 2.5132866428302836,
    13: 20.037841544639985,
    14: 0.26519072167008,
}

In [ ]:
####################################################################################################################################################################
# Bash and Google Cloud Bucket settings
####################################################################################################################################################################
# Specify the necessary arguments to upload the files to a Cloud Storage bucket
# I.e., create bash variables in order to create/check/delete Earth Engine Assets

# Specify main bash functions being used
bashFunction_EarthEngine = '/Users/justinstewart/opt/miniconda3/envs/gee_env/bin/earthengine'
bashFunctionGSUtil = '/Users/justinstewart/google-cloud-sdk/bin/gsutil'
#bashFunctionGSUtil = '/Users/justinstewart/exec -l /bin/bash/google-cloud-sdk/bin/gsutil'

# Specify the arguments to these functions
arglist_preEEUploadTable = ['upload','table']
arglist_postEEUploadTable = ['--x_column', longString, '--y_column', latString]
arglist_initEEUploadTable = ['--x_column', longString, '--y_column', latString, '--crs', CRStoUse]
arglist_preGSUtilUploadFile = ['cp']
formattedBucketOI = 'gs://'+bucketOfInterest
assetIDStringPrefix = '--asset_id='
arglist_CreateCollection = ['create','collection']
arglist_CreateFolder = ['create','folder']
arglist_Detect = ['asset','info']
arglist_Delete = ['rm','-r']
arglist_ls = ['ls']
stringsOfInterest = ['Asset does not exist or is not accessible']

# Compose the arguments into lists that can be run via the subprocess module
bashCommandList_Detect = [bashFunction_EarthEngine]+arglist_Detect
bashCommandList_Delete = [bashFunction_EarthEngine]+arglist_Delete
bashCommandList_ls = [bashFunction_EarthEngine]+arglist_ls
bashCommandList_CreateCollection = [bashFunction_EarthEngine]+arglist_CreateCollection
bashCommandList_CreateFolder = [bashFunction_EarthEngine]+arglist_CreateFolder

In [ ]:
####################################################################################################################################################################
# Helper functions
####################################################################################################################################################################

# Function to convert FeatureCollection to Image
def fcToImg(f):
	# Reduce to image, take mean per pixel
	img = sampledFC.reduceToImage(
		properties = [f],
		reducer = ee.Reducer.mean()
	)
	return img

# Function to convert GEE FC to pd.DataFrame. Not ideal as it's calling .getInfo(), but does the job
def GEE_FC_to_pd(fc):
    result = []

    values = fc.toList(500000).getInfo()

    BANDS = fc.first().propertyNames().getInfo()

    if 'system:index' in BANDS: BANDS.remove('system:index')

    for item in values:
        values_item = item['properties']
        row = [values_item[key] for key in BANDS]
        result.append(row)

    df = pd.DataFrame([item for item in result], columns = BANDS)
    df.replace('None', np.nan, inplace = True)
    return df

# Add point coordinates to FC as properties
def addLatLon(f):
    lat = f.geometry().coordinates().get(1)
    lon = f.geometry().coordinates().get(0)
    return f.set(latString, lat).set(longString, lon)

# R^2 function
def coefficientOfDetermination(fcOI,propertyOfInterest,propertyOfInterest_Predicted):
    # Compute the mean of the property of interest
    propertyOfInterestMean = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).select([propertyOfInterest]).reduceColumns(ee.Reducer.mean(),[propertyOfInterest])).get('mean'))

    # Compute the total sum of squares
    def totalSoSFunction(f):
        return f.set('Difference_Squared',ee.Number(ee.Feature(f).get(propertyOfInterest)).subtract(propertyOfInterestMean).pow(ee.Number(2)))
    totalSumOfSquares = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).map(totalSoSFunction).select(['Difference_Squared']).reduceColumns(ee.Reducer.sum(),['Difference_Squared'])).get('sum'))

    # Compute the residual sum of squares
    def residualSoSFunction(f):
        return f.set('Residual_Squared',ee.Number(ee.Feature(f).get(propertyOfInterest)).subtract(ee.Number(ee.Feature(f).get(propertyOfInterest_Predicted))).pow(ee.Number(2)))
    residualSumOfSquares = ee.Number(ee.Dictionary(ee.FeatureCollection(fcOI).map(residualSoSFunction).select(['Residual_Squared']).reduceColumns(ee.Reducer.sum(),['Residual_Squared'])).get('sum'))

    # Finalize the calculation
    r2 = ee.Number(1).subtract(residualSumOfSquares.divide(totalSumOfSquares))

    return ee.Number(r2)

# RMSE function
def RMSE(fcOI,propertyOfInterest,propertyOfInterest_Predicted):
    # Compute the squared difference between observed and predicted
    def propDiff(f):
        diff = ee.Number(f.get(propertyOfInterest)).subtract(ee.Number(f.get(propertyOfInterest_Predicted)))

        return f.set('diff', diff.pow(2))

    # calculate RMSE from squared difference _ ERROR HERE, calculate SQRT by hand after
    rmse = ee.Number(fcOI.map(propDiff).reduceColumns(ee.Reducer.mean(), ['diff']).get('mean')) #.sqrt()

    return rmse

# MAE function
def MAE(fcOI,propertyOfInterest,propertyOfInterest_Predicted):
    # Compute the absolute difference between observed and predicted
    def propDiff(f):
        diff = ee.Number(f.get(propertyOfInterest)).subtract(ee.Number(f.get(propertyOfInterest_Predicted)))

        return f.set('diff', diff.abs())

    # calculate MAE from squared difference
    MAE = ee.Number(fcOI.map(propDiff).reduceColumns(ee.Reducer.mean(), ['diff']).get('mean'))

    return MAE

# Function to take a feature with a classifier of interest
def computeCVAccuracyAndRMSE(featureWithClassifier):
    # Pull the classifier from the feature
    cOI = ee.Classifier(featureWithClassifier.get('c'))

    # Get the model type
    modelType = cOI.mode().getInfo()

    # Create a function to map through the fold assignments and compute the overall accuracy
    # for all validation folds
    def computeAccuracyForFold(foldFeature):
        # Organize the training and validation data

        foldNumber = ee.Number(ee.Feature(foldFeature).get('Fold'))
        trainingData_Random = fcOI.filterMetadata(cvFoldString_Random,'not_equals',foldNumber)
        validationData_Random = fcOI.filterMetadata(cvFoldString_Random,'equals',foldNumber)

        trainingData_Spatial = fcOI.filterMetadata(cvFoldString_Spatial,'not_equals',foldNumber)
        validationData_Spatial = fcOI.filterMetadata(cvFoldString_Spatial,'equals',foldNumber)

        # Train the classifier and classify the validation dataset
        trainedClassifier_Random = cOI.train(trainingData_Random,classProperty,covariateList)
        outputtedPropName_Random = classProperty+'_Predicted_Random'
        classifiedValidationData_Random = validationData_Random.classify(trainedClassifier_Random,outputtedPropName_Random)

        trainedClassifier_Spatial = cOI.train(trainingData_Spatial,classProperty,covariateList)
        outputtedPropName_Spatial = classProperty+'_Predicted_Spatial'
        classifiedValidationData_Spatial = validationData_Spatial.classify(trainedClassifier_Spatial,outputtedPropName_Spatial)

        if modelType == 'CLASSIFICATION':
            # Compute the overall accuracy of the classification
            errorMatrix_Random = classifiedValidationData_Random.errorMatrix(classProperty,outputtedPropName_Random,categoricalLevels)
            overallAccuracy_Random = ee.Number(errorMatrix_Random.accuracy())

            errorMatrix_Spatial = classifiedValidationData_Spatial.errorMatrix(classProperty,outputtedPropName_Spatial,categoricalLevels)
            overallAccuracy_Spatial = ee.Number(errorMatrix_Spatial.accuracy())
            return foldFeature.set('overallAccuracy_Random',overallAccuracy_Random).set('overallAccuracy_Spatial',overallAccuracy_Spatial)
        
        if modelType == 'REGRESSION':
            # Compute accuracy metrics
            r2ToSet_Random = coefficientOfDetermination(classifiedValidationData_Random,classProperty,outputtedPropName_Random)
            rmseToSet_Random = RMSE(classifiedValidationData_Random,classProperty,outputtedPropName_Random)
            maeToSet_Random = MAE(classifiedValidationData_Random,classProperty,outputtedPropName_Random)

            r2ToSet_Spatial = coefficientOfDetermination(classifiedValidationData_Spatial,classProperty,outputtedPropName_Spatial)
            rmseToSet_Spatial = RMSE(classifiedValidationData_Spatial,classProperty,outputtedPropName_Spatial)
            maeToSet_Spatial = MAE(classifiedValidationData_Spatial,classProperty,outputtedPropName_Spatial)
            return foldFeature.set('R2_Random',r2ToSet_Random).set('RMSE_Random', rmseToSet_Random).set('MAE_Random', maeToSet_Random)\
                                .set('R2_Spatial',r2ToSet_Spatial).set('RMSE_Spatial', rmseToSet_Spatial).set('MAE_Spatial', maeToSet_Spatial)

    # Compute the mean and std dev of the accuracy values of the classifier across all folds
    accuracyFC = kFoldAssignmentFC.map(computeAccuracyForFold)

    if modelType == 'REGRESSION':
        meanAccuracy_Random = accuracyFC.aggregate_mean('R2_Random')
        tsdAccuracy_Random = accuracyFC.aggregate_total_sd('R2_Random')
        meanAccuracy_Spatial = accuracyFC.aggregate_mean('R2_Spatial')
        tsdAccuracy_Spatial = accuracyFC.aggregate_total_sd('R2_Spatial')

        # Calculate mean and std dev of RMSE
        RMSEvals_Random = accuracyFC.aggregate_array('RMSE_Random')
        RMSEvalsSquared_Random = RMSEvals_Random.map(lambda f: ee.Number(f).multiply(f))
        sumOfRMSEvalsSquared_Random = RMSEvalsSquared_Random.reduce(ee.Reducer.sum())
        meanRMSE_Random = ee.Number.sqrt(ee.Number(sumOfRMSEvalsSquared_Random).divide(k))
        RMSEvals_Spatial = accuracyFC.aggregate_array('RMSE_Spatial')
        RMSEvalsSquared_Spatial = RMSEvals_Spatial.map(lambda f: ee.Number(f).multiply(f))
        sumOfRMSEvalsSquared_Spatial = RMSEvalsSquared_Spatial.reduce(ee.Reducer.sum())
        meanRMSE_Spatial = ee.Number.sqrt(ee.Number(sumOfRMSEvalsSquared_Spatial).divide(k))

        RMSEdiff_Random = accuracyFC.aggregate_array('RMSE_Random').map(lambda f: ee.Number(ee.Number(f).subtract(meanRMSE_Random)).pow(2))
        sumOfRMSEdiff_Random = RMSEdiff_Random.reduce(ee.Reducer.sum())
        sdRMSE_Random = ee.Number.sqrt(ee.Number(sumOfRMSEdiff_Random).divide(k))
        RMSEdiff_Spatial = accuracyFC.aggregate_array('RMSE_Spatial').map(lambda f: ee.Number(ee.Number(f).subtract(meanRMSE_Spatial)).pow(2))
        sumOfRMSEdiff_Spatial = RMSEdiff_Spatial.reduce(ee.Reducer.sum())
        sdRMSE_Spatial = ee.Number.sqrt(ee.Number(sumOfRMSEdiff_Spatial).divide(k))

        # Calculate mean and std dev of MAE
        meanMAE_Random = accuracyFC.aggregate_mean('MAE_Random')
        tsdMAE_Random= accuracyFC.aggregate_total_sd('MAE_Random')
        meanMAE_Spatial = accuracyFC.aggregate_mean('MAE_Spatial')
        tsdMAE_Spatial= accuracyFC.aggregate_total_sd('MAE_Spatial')

        # Compute the feature to return
        featureToReturn = featureWithClassifier.select(['cName']).set('Mean_R2_Random',meanAccuracy_Random,'StDev_R2_Random',tsdAccuracy_Random, 'Mean_RMSE_Random',meanRMSE_Random,'StDev_RMSE_Random',sdRMSE_Random, 'Mean_MAE_Random',meanMAE_Random,'StDev_MAE_Random',tsdMAE_Random)\
                                                                .set('Mean_R2_Spatial',meanAccuracy_Spatial,'StDev_R2_Spatial',tsdAccuracy_Spatial, 'Mean_RMSE_Spatial',meanRMSE_Spatial,'StDev_RMSE_Spatial',sdRMSE_Spatial, 'Mean_MAE_Spatial',meanMAE_Spatial,'StDev_MAE_Spatial',tsdMAE_Spatial)

    if modelType == 'CLASSIFICATION':
        accuracyFC_Random = kFoldAssignmentFC.map(computeAccuracyForFold)
        meanAccuracy_Random = accuracyFC_Random.aggregate_mean('overallAccuracy_Random')
        tsdAccuracy_Random = accuracyFC_Random.aggregate_total_sd('overallAccuracy_Random')
        accuracyFC_Spatial = kFoldAssignmentFC.map(computeAccuracyForFold)
        meanAccuracy_Spatial = accuracyFC_Spatial.aggregate_mean('overallAccuracy_Spatial')
        tsdAccuracy_Spatial = accuracyFC_Spatial.aggregate_total_sd('overallAccuracy_Spatial')

        # Compute the feature to return
        featureToReturn = featureWithClassifier.select(['cName']).set('Mean_overallAccuracy_Random',meanAccuracy_Random,'StDev_overallAccuracy_Random',tsdAccuracy_Random)\
                                                                .set('Mean_overallAccuracy_Spatial',meanAccuracy_Spatial,'StDev_overallAccuracy_Spatial',tsdAccuracy_Spatial)  

    return featureToReturn


# Function to add folds stratified per biome
def assignFolds(biome):
	fc_filtered = fc_agg.filter(ee.Filter.eq(stratificationVariableString, biome))

	cvFoldsToAssign = ee.List.sequence(0, fc_filtered.size()).map(lambda i: ee.Number(i).mod(k).add(1))

	fc_sorted = fc_filtered.randomColumn(seed = biome).sort('random')

	fc_wCVfolds = ee.FeatureCollection(cvFoldsToAssign.zip(fc_sorted.toList(fc_filtered.size())).map(lambda f: ee.Feature(ee.List(f).get(1)).set(cvFoldString, ee.List(f).get(0))))

	return fc_wCVfolds

In [ ]:
# Data processing ####################################################################################################################################################################
# Import raw data
pathOfPointCollection = holdingFolder+'/'+titleOfRawPointCollection+'.csv'
# Load rawPointCollection
rawPointCollection = pd.read_csv(pathOfPointCollection, float_precision='round_trip')
print('Size of original Collection', rawPointCollection.shape[0])

# Rename classification property column
rawPointCollection.rename(columns={'rarefied': classProperty}, inplace=True)

# Shuffle the data frame while setting a new index to ensure geographic clumps of points are not clumped in any way
fcToAggregate = rawPointCollection.sample(frac = 1, random_state = 42).reset_index(drop=True)

# Remove duplicates / pixel aggregate
#preppedCollection = fcToAggregate.drop_duplicates(subset = covariateList+[classProperty], keep = 'first')[['sample_id']+covariateList+["Resolve_Biome"]+[classProperty]+['Pixel_Lat', 'Pixel_Long']]
preppedCollection = fcToAggregate # try keeping all values

print('Number of aggregated pixels', preppedCollection.shape[0])

# Drop NAs
preppedCollection = preppedCollection.dropna(how='any')
print('After dropping NAs', preppedCollection.shape[0])

# Log transform classProperty; if specified
if log_transform_classProperty == True:
    preppedCollection[classProperty] = np.log(preppedCollection[classProperty] + 1)

# Convert biome column to int, to correct odd rounding errors
preppedCollection[stratificationVariableString] = preppedCollection[stratificationVariableString].astype(int)

# Generate random folds, stratified by biome
preppedCollection[cvFoldString_Random] = (preppedCollection.groupby('Resolve_Biome').cumcount() % k) + 1

# Write the CSV to disk and upload it to Earth Engine as a Feature Collection
localPathToCVAssignedData = holdingFolder+'/'+titleOfCSVWithCVAssignments+'.csv'
preppedCollection.to_csv(localPathToCVAssignedData,index=False)



try:
    # try whether fcOI is present
    fcOI = ee.FeatureCollection('users/'+usernameFolderString+'/'+projectFolder+'/'+titleOfCSVWithCVAssignments)

    # Print size of dataset
    print(guild, fcOI.size().getInfo())

except Exception as e:
    # Format the bash call to upload the file to the Google Cloud Storage bucket
    gsutilBashUploadList = [bashFunctionGSUtil]+arglist_preGSUtilUploadFile+[localPathToCVAssignedData]+[formattedBucketOI]
    subprocess.run(gsutilBashUploadList)
    print(titleOfCSVWithCVAssignments+' uploaded to a GCSB!')

    # Wait for a short period to ensure the command has been received by the server
    time.sleep(normalWaitTime/2)

    # Wait for the GSUTIL uploading process to finish before moving on
    while not all(x in subprocess.run([bashFunctionGSUtil,'ls',formattedBucketOI],stdout=subprocess.PIPE).stdout.decode('utf-8') for x in [titleOfCSVWithCVAssignments]):
        print('Not everything is uploaded...')
        time.sleep(normalWaitTime)
    print('Everything is uploaded; moving on...')

    # Upload the file into Earth Engine as a table asset
    assetIDForCVAssignedColl = 'users/'+usernameFolderString+'/'+projectFolder+'/'+titleOfCSVWithCVAssignments
    earthEngineUploadTableCommands = [bashFunction_EarthEngine]+arglist_preEEUploadTable+[assetIDStringPrefix+assetIDForCVAssignedColl]+[formattedBucketOI+'/'+titleOfCSVWithCVAssignments+'.csv']+arglist_postEEUploadTable
    subprocess.run(earthEngineUploadTableCommands)
    print('Upload to EE queued!')

    # Wait for a short period to ensure the command has been received by the server
    time.sleep(normalWaitTime/2)

    # !! Break and wait
    count = 1
    while count >= 1:
        taskList = [str(i) for i in ee.batch.Task.list()]
        subsetList = [s for s in taskList if classProperty in s]
        subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
        count = len(subsubList)
        print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Number of running jobs:', count)
        time.sleep(normalWaitTime)
    print('Moving on...')


In [ ]:
# Hyperparameter tuning
# HELPFUL FUNCTION earthengine create folder users/justin618s/DensityResubmit/Hyphae/hyperparameter_tuning 

fcOI = ee.FeatureCollection('users/'+usernameFolderString+'/'+projectFolder+'/'+titleOfCSVWithCVAssignments)
print(guild + ' hyperparameter tuning')

# Define hyperparameters for grid search
varsPerSplit_list = list(range(4,14,2))
leafPop_list = list(range(2,14,2))

classifierList = []
# Create list of classifiers for regression
for vps in varsPerSplit_list:
    for lp in leafPop_list:
        model_name = classProperty + '_rf_VPS' + str(vps) + '_LP' + str(lp) + '_REGRESSION'
        rf = ee.Feature(ee.Geometry.Point([0,0])).set('cName',model_name,'c',ee.Classifier.smileRandomForest(
        numberOfTrees = nTrees,
        variablesPerSplit = vps,
        minLeafPopulation = lp,
        bagFraction = 0.632,
        seed = 42
        ).setOutputMode('REGRESSION'))

        classifierList.append(rf)


# # If grid search was not performed yet:
# Make a feature collection from the k-fold assignment list
kFoldAssignmentFC = ee.FeatureCollection(ee.List(kList).map(lambda n: ee.Feature(ee.Geometry.Point([0,0])).set('Fold',n)))

# Check if any models have been completed
try:
    grid_search_results = ee.FeatureCollection('users/'+usernameFolderString+'/'+projectFolder+'/'+classProperty+'_grid_search_results')
    print(grid_search_results.size().getInfo())

except Exception as e:
    try:
        # Create list of finished models
        finished_models = subprocess.run(bashCommandList_ls+['users/'+usernameFolderString+'/'+projectFolder+'/hyperparameter_tuning/'], stdout=subprocess.PIPE).stdout.splitlines()
        finished_models = [model.decode('ascii').split('/')[-1] for model in finished_models]
        
    except Exception as e:
        finished_models = list()

    # Perform model testing for remaining hyperparameter settings
    for rf in classifierList:
        if rf.get('cName').getInfo() in finished_models:
            print('Model', classifierList.index(rf), 'out of total of', len(classifierList), 'already finished')
        else:
            print('Testing model', classifierList.index(rf), 'out of total of', len(classifierList))
            fcOI = ee.FeatureCollection('users/'+usernameFolderString+'/'+projectFolder+'/'+titleOfCSVWithCVAssignments)
            accuracy_feature = ee.Feature(computeCVAccuracyAndRMSE(rf))
            accuracy_featureExport = ee.batch.Export.table.toAsset(
                collection = ee.FeatureCollection([accuracy_feature]),
                description = classProperty+rf.get('cName').getInfo(),
                assetId = 'users/'+usernameFolderString+'/'+projectFolder+'/hyperparameter_tuning/'+rf.get('cName').getInfo())
            accuracy_featureExport.start()

    # !! Break and wait
    count = 1
    while count >= 1:
        taskList = [str(i) for i in ee.batch.Task.list()]
        subsetList = [s for s in taskList if classProperty in s]
        subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
        count = len(subsubList)
        print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Number of running jobs:', count)
        time.sleep(normalWaitTime)
    print('Moving on...')

# Fetch FC from GEE
grid_search_results = ee.FeatureCollection([ee.FeatureCollection('users/'+usernameFolderString+'/'+projectFolder+'/hyperparameter_tuning/'+rf.get('cName').getInfo()) for rf in classifierList]).flatten()
classDf = GEE_FC_to_pd(grid_search_results)

grid_search_results_export = ee.batch.Export.table.toAsset(
    collection = grid_search_results,
    description = classProperty+'_grid_search_results',
    assetId = 'users/'+usernameFolderString+'/'+projectFolder+'/'+classProperty+'_grid_search_results')
grid_search_results_export.start()

# Sort values
classDfSorted = classDf.sort_values([sort_acc_prop], ascending = False)

# Write model results to csv
classDfSorted.to_csv('output/'+today+'_'+classProperty+'_grid_search_results.csv', index=False)

# Get top model name
bestModelName = grid_search_results.limit(1, sort_acc_prop, False).first().get('cName')

# Get top 10 models
top_10Models = grid_search_results.limit(10, sort_acc_prop, False).aggregate_array('cName')

print('Moving on...')

In [ ]:
# Predicted - Observed
top_10Models = grid_search_results.limit(10, sort_acc_prop, False).aggregate_array('cName')
bestModelName = grid_search_results.limit(1, sort_acc_prop, False).first().get('cName')
fcOI = ee.FeatureCollection('users/'+usernameFolderString+'/'+projectFolder+'/'+titleOfCSVWithCVAssignments)
print('starting pred-obs')

def predObs(fcOI):
    if ensemble == False:
        # Load the best model from the classifier list
        classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', bestModelName).first()).get('c'))

        # Train the classifier with the collection
        trainedClassifier = classifier.train(fcOI, classProperty, covariateList)

        # Classify the FC
        classifiedFC = fcOI.classify(trainedClassifier,classProperty+'_Predicted')

        return classifiedFC

    if ensemble == True:
        def classifyFC(classifierName):
            # Load the best model from the classifier list
            classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', classifierName).first()).get('c'))

            # Train the classifier with the collection
            trainedClassifier = classifier.train(fcOI, classProperty, covariateList)

            # Classify the FC
            classifiedFC = fcOI.classify(trainedClassifier,classProperty+'_Predicted')

            return classifiedFC

        # Map function over list of models in ensemble
        classifiedFC = ee.FeatureCollection(top_10Models.map(classifyFC)).flatten()

        return classifiedFC

# Classify FC
predObs = predObs(fcOI)

# Add coordinates to FC
predObs = predObs.map(addLatLon)

# reverse log transform predicted and observed values
if log_transform_classProperty == True:
    predObs = predObs.map(lambda f: f.set(classProperty, ee.Number(f.get(classProperty)).exp().subtract(1)))
    predObs = predObs.map(lambda f: f.set(classProperty+'_Predicted', ee.Number(f.get(classProperty+'_Predicted')).exp().subtract(1)))

# Add residuals to FC
predObs_wResiduals = predObs.map(lambda f: f.set('AbsResidual', ee.Number(f.get(classProperty+'_Predicted')).subtract(f.get(classProperty)).abs()))

# Export to Assets
predObsexport = ee.batch.Export.table.toAsset(
    collection = predObs_wResiduals,
    description = classProperty+'_pred_obs',
    assetId = 'users/'+usernameFolderString+'/'+projectFolder+'/'+classProperty+'_pred_obs'
)
predObsexport.start()

# !! Break and wait
count = 1
while count >= 1:
    taskList = [str(i) for i in ee.batch.Task.list()]
    subsetList = [s for s in taskList if classProperty in s]
    subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
    count = len(subsubList)
    print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Waiting for pred/obs to complete...', end = '\r')
    time.sleep(normalWaitTime)
print('Moving on...')

predObs_wResiduals = ee.FeatureCollection('users/'+usernameFolderString+'/'+projectFolder+'/'+classProperty+'_pred_obs')

# Convert to pd
predObs_df = GEE_FC_to_pd(predObs_wResiduals)

# Group by sample ID to return mean across ensemble prediction
predObs_df = pd.DataFrame(predObs_df.groupby('sample_id').mean().to_records())

predObs_df.to_csv('output/'+today+'_'+classProperty+'_pred_obs.csv')


In [ ]:
# Classify image

#reference levels
GrowthFormKnown = ee.Image.constant(0) # Ref level (most common class is unknown)
Growth_Form_Grass = ee.Image.constant(1) # Ref level (most common class)
Growth_Form_Herb = ee.Image.constant(0)
Growth_Form_Shrub = ee.Image.constant(0) 
Growth_Form_Tree = ee.Image.constant(0)
cultivated1_wild0_Pironon2023 = ee.Image.constant(1)# Ref level
Season_NorthernHemHarmonized_Spring = ee.Image.constant(1) #Reference Level (all seasons averaged)
Season_NorthernHemHarmonized_Summer = ee.Image.constant(1) #Reference Level (all seasons averaged)
Season_NorthernHemHarmonized_Autumn = ee.Image.constant(1) #Reference Level (all seasons averaged)
Season_NorthernHemHarmonized_Winter = ee.Image.constant(1) #Reference Level (all seasons averaged)
Season_NorthernHemHarmonized_NA = ee.Image.constant(0)
sample_size = ee.Image.constant(9) #<- Reference Level (avg)
soil_depth_depth_cm = ee.Image.constant(15) # Reference Level (desired depth)
soil_core_volume_cm3 = ee.Image.constant(330) #Reference Level (avg)
sampling_intensity = ee.Image.constant(1).unmask(0) #Equal sampling scenario (avg)

constant_imgs = ee.ImageCollection.fromImages([
  GrowthFormKnown,
  Growth_Form_Grass,
  Growth_Form_Herb,
  Growth_Form_Shrub,
  Growth_Form_Tree,
  cultivated1_wild0_Pironon2023,
  Season_NorthernHemHarmonized_Spring,
  Season_NorthernHemHarmonized_Summer,
  Season_NorthernHemHarmonized_Autumn,
  Season_NorthernHemHarmonized_Winter,
  Season_NorthernHemHarmonized_NA,
  soil_core_volume_cm3,
  sample_size,
  soil_depth_depth_cm,
  sampling_intensity
]).toBands().rename([
  "GrowthFormKnown",
  "Growth_Form_Grass",
  "Growth_Form_Herb",
  "Growth_Form_Shrub",
  "Growth_Form_Tree",
  "cultivated1_wild0_Pironon2023",
  "Season_NorthernHemHarmonized_Spring",
  "Season_NorthernHemHarmonized_Summer",
  "Season_NorthernHemHarmonized_Autumn",
  "Season_NorthernHemHarmonized_Winter",
  "Season_NorthernHemHarmonized_NA",
  "soil_core_volume_cm3",
  "sample_size",
  "soil_depth_depth_cm",
  "sampling_intensity"
])


def finalImageClassification(compositeImg):
    if ensemble == False:
        # Load the best model from the classifier list
        classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', bestModelName).first()).get('c'))

        # Train the classifier with the collection
        trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

        # Classify the FC
        classifiedImage = compositeImg.classify(trainedClassifer,classProperty+'_Predicted')

        return classifiedImage

    if ensemble == True:
        def classifyImage(modelName):
            # Load the best model from the classifier list
            classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', modelName).first()).get('c'))

            # Train the classifier with the collection
            trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

            # Classify the FC
            classifiedImage = compositeImg.classify(trainedClassifer,classProperty+'_Predicted')

            return classifiedImage

        # Classify the images, return mean
        classifiedImage = ee.ImageCollection(top_10Models.map(classifyImage)).mean()
    return classifiedImage

# Create appropriate composite image with bands to use
compositeToClassify = compositeOfInterest.addBands(constant_imgs).select(covariateList).reproject(compositeOfInterest.projection())
classifiedImage = finalImageClassification(compositeToClassify)

if log_transform_classProperty == True:
     classifiedImage = classifiedImage.exp().subtract(1)

classifiedImageExport = ee.batch.Export.image.toAsset(
    image = classifiedImage.toFloat(),
     description = classProperty+'_ClassifiedImage',
     assetId = 'users/'+usernameFolderString+'/'+projectFolder+'/'+classProperty+'_ClassifiedImage',
     crs = 'EPSG:4326',
     crsTransform = '[0.08333333333333333,0,-180,0,-0.08333333333333333,90]',
     region = exportingGeometry,
     maxPixels = int(1e13),
     pyramidingPolicy = {".default": pyramidingPolicy}
 )
classifiedImageExport.start()
print("Moving on...")

In [ ]:
# Variable importance metrics
print('Calculating variable importance')
import os
downloads_path = os.path.expanduser("~/Downloads")  ### CHANGED: Used correctly below

if ensemble == False:
    classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', bestModelName).first()).get('c'))

    # Train the classifier with the collection
    trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

    # Get the feature importance from the trained classifier and write to a .csv file and as a bar plot as .png file
    featureImportances = trainedClassifer.explain().get('importance').getInfo()

    featureImportances = pd.DataFrame(featureImportances.items(),
                                        columns=['Variable', 'Feature_Importance']).sort_values(by='Feature_Importance',
                                                                                                ascending=False)

    # Scale values
    featureImportances['Feature_Importance'] = featureImportances['Feature_Importance'] - featureImportances['Feature_Importance'].min()
    featureImportances['Feature_Importance'] = featureImportances['Feature_Importance'] / featureImportances['Feature_Importance'].max()

if ensemble == True:
    # Instantiate empty dataframe
    featureImportances = pd.DataFrame(columns=['Variable', 'Feature_Importance'])

    for i in list(range(0,10)):
        classifierName = top_10Models.get(i)
        classifier = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', classifierName).first()).get('c'))

        # Train the classifier with the collection
        trainedClassifer = classifier.train(fcOI, classProperty, covariateList)

        # Get the feature importance from the trained classifier and write to a .csv file and as a bar plot as .png file
        featureImportancesToAdd = trainedClassifer.explain().get('importance').getInfo()
        featureImportancesToAdd = pd.DataFrame(featureImportancesToAdd.items(),
                                            columns=['Variable', 'Feature_Importance']).sort_values(by='Feature_Importance',
                                                                                                    ascending=False)
        # Scale values
        featureImportancesToAdd['Feature_Importance'] = featureImportancesToAdd['Feature_Importance'] - featureImportancesToAdd['Feature_Importance'].min()
        featureImportancesToAdd['Feature_Importance'] = featureImportancesToAdd['Feature_Importance'] / featureImportancesToAdd['Feature_Importance'].max()

        featureImportances = pd.concat([featureImportances, featureImportancesToAdd])

    featureImportances = pd.DataFrame(featureImportances.groupby('Variable').mean().to_records())

# Write to csv
featureImportances.sort_values('Feature_Importance', ascending = False ,inplace = True)
featureImportances.to_csv(
    os.path.join(downloads_path, f"{today}_{classProperty}_featureImportances.csv")  ### CHANGED
)

# Create and save plot
plt = featureImportances[:10].plot(x='Variable', y='Feature_Importance', kind='bar', legend=False,
                                title='Feature Importances')
fig = plt.get_figure()
fig.savefig(
    os.path.join(downloads_path, f"{today}_{classProperty}_FeatureImportances.png"), bbox_inches='tight'  ### CHANGED
)

print('Variable importance metrics complete! Moving on...')


In [ ]:
# Bootstrapping
# Input the number of points to use for each bootstrap model: equal to number of observations in training dataset
print("Starting bootstrapping")
bootstrapModelSize = preppedCollection.shape[0]

# Run a for loop to create multiple bootstrap iterations and upload them to the Google Cloud Storage Bucket
# Create an empty list to store all of the file name strings being uploaded (for later use) 
fileNameList = []
stratSample = preppedCollection.head(0)

for n in seedsToUseForBootstrapping:
    # Perform the subsetting
    sampleToConcat = preppedCollection.groupby(stratificationVariableString, group_keys=False).apply(
    lambda x: x.sample(n=len(x), replace=True, random_state=n))

   #sampleToConcat = preppedCollection.groupby(stratificationVariableString, group_keys=False).apply(lambda x: x.sample(n=int(round((strataDict.get(x.name)/100)*bootstrapModelSize)), replace=True, random_state=n))
    sampleToConcat['bootstrapIteration'] = n
    stratSample = pd.concat([stratSample, sampleToConcat])

# Format the title of the CSV and export it to a holding location
fullLocalPath = holdingFolder+'/'+bootstrapSamples+'.csv'
stratSample.to_csv(holdingFolder+'/'+bootstrapSamples+'.csv',index=False)

# Format the bash call to upload the files to the Google Cloud Storage bucket
gsutilBashUploadList = [bashFunctionGSUtil]+arglist_preGSUtilUploadFile+[fullLocalPath]+[formattedBucketOI]
subprocess.run(gsutilBashUploadList)
print(bootstrapSamples+' uploaded to a GCSB!')

# Wait for the GSUTIL uploading process to finish before moving on
while not all(x in subprocess.run([bashFunctionGSUtil,'ls',formattedBucketOI],stdout=subprocess.PIPE).stdout.decode('utf-8') for x in [bootstrapSamples]):
    print('Not everything is uploaded...')
    time.sleep(5)
print('Everything is uploaded moving on...')

# Upload the file into Earth Engine as a table asset
assetIDForCVAssignedColl = 'users/'+usernameFolderString+'/'+projectFolder+'/'+bootstrapSamples
earthEngineUploadTableCommands = [bashFunction_EarthEngine]+arglist_preEEUploadTable+[assetIDStringPrefix+assetIDForCVAssignedColl]+[formattedBucketOI+'/'+bootstrapSamples+'.csv']+arglist_postEEUploadTable
subprocess.run(earthEngineUploadTableCommands)
print('Upload to EE queued!')

# Wait for a short period to ensure the command has been received by the server
time.sleep(normalWaitTime/2)

# !! Break and wait
count = 1
while count >= 1:
    taskList = [str(i) for i in ee.batch.Task.list()]
    subsetList = [s for s in taskList if classProperty in s]
    subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
    count = len(subsubList)
    print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Number of running jobs:', count)
    time.sleep(normalWaitTime)
print('Moving on...')

# Load the best model from the classifier list
classifierToBootstrap = ee.Classifier(ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName','equals',bestModelName).first()).get('c'))
print('Loading model')
# Create empty list to store all fcs
fcList = []
# Run a for loop to create multiple bootstrap iterations
for n in seedsToUseForBootstrapping:
    #
    collectionPath = 'users/'+usernameFolderString+'/'+projectFolder+'/'+bootstrapSamples

    # Load the collection from the path
    fcToTrain = ee.FeatureCollection(collectionPath).filter(ee.Filter.eq('bootstrapIteration', n))

    # Append fc to list
    fcList.append(fcToTrain)

# Helper fucntion to train a RF classifier and classify the composite image
def bootstrapFunc(fc):
    # Train the classifier with the collection
    trainedClassifer = classifierToBootstrap.train(fc, classProperty, covariateList)

    # Classify the image
    # added in below line 
    compositeToClassify = compositeOfInterest.addBands(constant_imgs).select(covariateList).reproject(compositeOfInterest.projection())
    classifiedImage = compositeToClassify.classify(trainedClassifer,classProperty+'_Predicted')

    return classifiedImage

# Reduce bootstrap images to mean
meanImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(
    reducer = ee.Reducer.mean()
)
#compositeToClassify.bandNames().getInfo()
print('Reducing image to CIs for mean')
# Reduce bootstrap images to lower and upper CIs
upperLowerCIImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(
    reducer = ee.Reducer.percentile([2.5,97.5],['lower','upper'])
)

# Reduce bootstrap images to standard deviation
stdDevImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(
    reducer = ee.Reducer.stdDev()
)

# Coefficient of Variation: stdDev divided by mean
coefOfVarImage = stdDevImage.divide(meanImage).rename('Bootstrapped_CoefOfVar')
print("Moving on...")

In [ ]:
# Bootstrapping part 2 ( if Already uploaded (e.g. you download and correct typos in column names))
import os
import subprocess
import time
import datetime
import ee

# Set file paths
fullLocalPath = os.path.join(holdingFolder, bootstrapSamples + '.csv')
gs_path = f"{formattedBucketOI}/{bootstrapSamples}.csv"
ee_asset_path = f'users/{usernameFolderString}/{projectFolder}/{bootstrapSamples}'

# Check if file exists in Google Cloud Storage
def check_gcs_file_exists(gs_path):
    result = subprocess.run([bashFunctionGSUtil, 'ls', gs_path], stdout=subprocess.PIPE)
    return gs_path in result.stdout.decode('utf-8')

# Check if Earth Engine asset exists
def check_ee_asset_exists(ee_asset_path):
    try:
        ee.data.getAsset(ee_asset_path)
        return True
    except:
        return False

# Step 1: Check if the GCS file needs to be uploaded
if not check_gcs_file_exists(gs_path):
    print("Starting bootstrapping")
    bootstrapModelSize = preppedCollection.shape[0]

    stratSample = preppedCollection.head(0)

    for n in seedsToUseForBootstrapping:
        # Perform the subsetting
        sampleToConcat = preppedCollection.groupby(stratificationVariableString, group_keys=False).apply(
            lambda x: x.sample(n=int(round((strataDict.get(x.name) / 100) * bootstrapModelSize)), replace=True, random_state=n))
        sampleToConcat['bootstrapIteration'] = n
        stratSample = pd.concat([stratSample, sampleToConcat])

    # Export to CSV
    stratSample.to_csv(fullLocalPath, index=False)
    print(f"{bootstrapSamples}.csv generated locally")

    # Upload the CSV file to GCS
    gsutilBashUploadList = [bashFunctionGSUtil] + arglist_preGSUtilUploadFile + [fullLocalPath] + [formattedBucketOI]
    subprocess.run(gsutilBashUploadList)
    print(f"{bootstrapSamples} uploaded to GCS!")

# Step 2: Ensure the GCS upload is complete before moving on
while not check_gcs_file_exists(gs_path):
    print("Waiting for GCS upload to complete...")
    time.sleep(5)
print("GCS upload complete. Moving on...")

# Step 3: Check if the EE asset needs to be uploaded
if not check_ee_asset_exists(ee_asset_path):
    # Upload to Earth Engine as a table asset
    assetIDForCVAssignedColl = f'users/{usernameFolderString}/{projectFolder}/{bootstrapSamples}'
    earthEngineUploadTableCommands = [
        bashFunction_EarthEngine,
        *arglist_preEEUploadTable,
        assetIDStringPrefix + assetIDForCVAssignedColl,
        gs_path,
        *arglist_postEEUploadTable
    ]
    subprocess.run(earthEngineUploadTableCommands)
    print('Upload to EE queued!')

    # Wait for EE upload to complete
    time.sleep(normalWaitTime / 2)

    count = 1
    while count >= 1:
        taskList = [str(i) for i in ee.batch.Task.list()]
        subsetList = [s for s in taskList if classProperty in s]
        subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
        count = len(subsubList)
        print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'), 'Number of running jobs:', count)
        time.sleep(normalWaitTime)
    print("EE asset upload complete. Moving on...")
else:
    print("EE asset already exists. Skipping EE upload.")

# Step 4: Proceed with bootstrapping if everything is uploaded
classifierToBootstrap = ee.Classifier(
    ee.Feature(ee.FeatureCollection(classifierList).filterMetadata('cName', 'equals', bestModelName).first()).get('c')
)
print("Loading model")

# Create empty list to store all feature collections (fcs)
fcList = []
for n in seedsToUseForBootstrapping:
    collectionPath = f'users/{usernameFolderString}/{projectFolder}/{bootstrapSamples}'
    fcToTrain = ee.FeatureCollection(collectionPath).filter(ee.Filter.eq('bootstrapIteration', n))
    fcList.append(fcToTrain)

# Helper function to train RF classifier and classify the composite image
def bootstrapFunc(fc):
    trainedClassifier = classifierToBootstrap.train(fc, classProperty, covariateList)
    compositeToClassify = compositeOfInterest.addBands(constant_imgs).select(covariateList).reproject(compositeOfInterest.projection())
    classifiedImage = compositeToClassify.classify(trainedClassifier, classProperty + '_Predicted')
    return classifiedImage

# Reduce bootstrap images to mean, CIs, and standard deviation
meanImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(ee.Reducer.mean())
upperLowerCIImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(ee.Reducer.percentile([2.5, 97.5], ['lower', 'upper']))
stdDevImage = ee.ImageCollection.fromImages(list(map(bootstrapFunc, fcList))).reduce(ee.Reducer.stdDev())
coefOfVarImage = stdDevImage.divide(meanImage).rename('Bootstrapped_CoefOfVar')

print("Bootstrapping complete. Moving on...")


In [ ]:
# Bootstrapping Part 3 (file already uploaded, generates individual rasters) 
classifierToBootstrap = ee.Classifier(
    ee.Feature(
        ee.FeatureCollection(classifierList)
        .filterMetadata('cName', 'equals', bestModelName)
        .first()
    ).get('c')
)
print("Loading model for bootstrapping")

collectionPath = f'users/{usernameFolderString}/{projectFolder}/{bootstrapSamples}'
bootstrapFolder = f'users/{usernameFolderString}/{projectFolder}/Bootstraps'

# Helper: chunk a list into groups of size n
def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# Check if asset exists
def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except Exception:
        return False

# Split bootstrap seeds into chunks of 10
seed_chunks = list(chunk_list(seedsToUseForBootstrapping, 10))

# Loop over chunks
for chunk_idx, chunk in enumerate(seed_chunks):
    print(f"=== Starting chunk {chunk_idx+1}/{len(seed_chunks)} with seeds {chunk} ===")
    
    for n in chunk:
        asset_id = f"{bootstrapFolder}/{classProperty}_chunk{chunk_idx+1}_bootstrap_{n}"
        
        if asset_exists(asset_id):
            print(f"Asset already exists, skipping: {asset_id}")
            continue
        
        fcToTrain = ee.FeatureCollection(collectionPath).filter(
            ee.Filter.eq('bootstrapIteration', n)
        )
        
        trainedClassifier = classifierToBootstrap.train(fcToTrain, classProperty, covariateList)
        
        classifiedImage = compositeOfInterest.addBands(constant_imgs).select(covariateList) \
            .classify(trainedClassifier, classProperty + '_Predicted')
        
        export = ee.batch.Export.image.toAsset(
            image = classifiedImage.toFloat(),
            description = f"{classProperty}_chunk{chunk_idx+1}_bootstrap_{n}",
            assetId = asset_id,
            region = exportingGeometry,
            scale = 1000,
            maxPixels = 1e13
        )
        export.start()
        print(f"Bootstrap {n} export queued at {asset_id}")
    
    # Wait for this chunk to finish before moving on
    print(f"Waiting for chunk {chunk_idx+1} to complete...")
    count = 1
    while count >= 1:
        taskList = [str(i) for i in ee.batch.Task.list()]
        subsetList = [s for s in taskList if classProperty in s and 'bootstrap' in s]
        subsubList = [s for s in subsetList if any(xs in s for xs in ['RUNNING', 'READY'])]
        count = len(subsubList)
        print(datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S'),
              f'Number of running bootstrap jobs in chunk {chunk_idx+1}:', count, end='\r')
        time.sleep(normalWaitTime)
    print(f"Chunk {chunk_idx+1} finished.")

print("All bootstrap exports finished. Moving on...")

# ------------------------
# Collect exported bootstraps into an ImageCollection
# ------------------------
bootstrapCollection = ee.ImageCollection([
    f"{bootstrapFolder}/{classProperty}_chunk{chunk_idx+1}_bootstrap_{n}"
    for chunk_idx, chunk in enumerate(seed_chunks)
    for n in chunk
    if asset_exists(f"{bootstrapFolder}/{classProperty}_chunk{chunk_idx+1}_bootstrap_{n}")
])

print("Bootstrap ImageCollection ready.")


In [ ]:
#Extrapolation analysis 
# Univariate int-ext analysis
# Function to perform extrapolation analysis using PCA
  
# List of all covariates you're trying to select
covariateList_PCA = [
    'CGIAR_PET', 'CHELSA_BIO_Annual_Mean_Temperature', 'CHELSA_BIO_Annual_Precipitation',
    'CHELSA_BIO_Max_Temperature_of_Warmest_Month', 'CHELSA_BIO_Precipitation_Seasonality',
    'EarthEnvTexture_CoOfVar_EVI', 'EarthEnvTexture_Correlation_EVI', 'EarthEnvTexture_Homogeneity_EVI',
    'EarthEnvTopoMed_AspectCosine', 'EarthEnvTopoMed_AspectSine', 'EarthEnvTopoMed_Elevation',
    'EarthEnvTopoMed_Slope', 'EarthEnvTopoMed_TopoPositionIndex', 'EsaCci_BurntAreasProbability',
    'GHS_Population_Density', 'harmonized-aboveground-biomass', 'harmonized-belowground-biomass',
    'MODIS_NPP', 'SG_Depth_to_bedrock', 'SG_SOC_Content_005cm', 'SG_Soil_pH_H2O_005cm', 'sampling_intensity',
    'phosphorus', 'nitrogen', 'isric-soil-proportion-of-sand',"EarthEnvLandcover-class-barren", 'earthEnvLandcover-class-cultivated-managed-vegetation',
    'earthEnvLandcover-class-evergreen-broadleaf-trees', 'earthEnvLandcover-class-evergreen-decid-needleleaf-trees',
    'earthEnvLandcover-class-mixed-other-trees', 'earthEnvLandcover-class-shrubs', 'earthEnvLandcover-class-herbaceous-vegetation',
    'earthEnvLandcover-class-regularly-flooded-vegetation', 'earthEnvLandcover-class-barren', 'myco_veg_cover'
]

# Filter out columns that aren't in your DataFrame
covariateList_PCA = [col for col in covariateList_PCA if col in preppedCollection.columns]


##################################################################################################################################################################
# Multivariate (PCA) int-ext analysis
##################################################################################################################################################################

# Input the proportion of variance that you would like to cover
propOfVariance = 90
# PCA interpolation/extrapolation helper function
def assessExtrapolation(fcOfInterest, propOfVariance, compositeToClassify):

##def assessExtrapolation(fcOfInterest, propOfVariance):
    # Compute the mean and standard deviation of each band, then standardize the point data
    meanVector = fcOfInterest.mean()
    stdVector = fcOfInterest.std()
    standardizedData = ((fcOfInterest-meanVector)/stdVector)

    # Then standardize the composite from which the points were sampled
    meanList = meanVector.tolist()
    stdList = stdVector.tolist()
    bandNames = list(meanVector.index)
    print(bandNames)
    meanImage = ee.Image(meanList).rename(bandNames)
    stdImage = ee.Image(stdList).rename(bandNames)
    compositeToClassify = compositeToClassify.select(bandNames)
    standardizedImage = compositeToClassify.subtract(meanImage).divide(stdImage)

    #standardizedImage = compositeToClassify.subtract(meanImage).divide(stdImage)

    # Run a PCA on the point samples
    pcaOutput = PCA()
    pcaOutput.fit(standardizedData)

    # Save the cumulative variance represented by each PC
    cumulativeVariance = np.cumsum(np.round(pcaOutput.explained_variance_ratio_, decimals=4)*100)

    # Make a list of PC names for future organizational purposes
    pcNames = ['PC'+str(x) for x in range(1,fcOfInterest.shape[1]+1)]

    # Get the PC loadings as a data frame
    loadingsDF = pd.DataFrame(pcaOutput.components_,columns=[str(x)+'_Loads' for x in bandNames],index=pcNames)

    # Get the original data transformed into PC space
    transformedData = pd.DataFrame(pcaOutput.fit_transform(standardizedData,standardizedData),columns=pcNames)

    # Make principal components images, multiplying the standardized image by each of the eigenvectors
    # Collect each one of the images in a single image collection

    # First step: make an image collection wherein each image is a PC loadings image
    listOfLoadings = ee.List(loadingsDF.values.tolist())
    eePCNames = ee.List(pcNames)
    zippedList = eePCNames.zip(listOfLoadings)
    def makeLoadingsImage(zippedValue):
        return ee.Image.constant(ee.List(zippedValue).get(1)).rename(bandNames).set('PC',ee.List(zippedValue).get(0))
    loadingsImageCollection = ee.ImageCollection(zippedList.map(makeLoadingsImage))

    # Second step: multiply each of the loadings image by the standardized image and reduce it using a "sum"
    # to finalize the matrix multiplication
    def finalizePCImages(loadingsImage):
        PCName = ee.String(ee.Image(loadingsImage).get('PC'))
        return ee.Image(loadingsImage).multiply(standardizedImage).reduce('sum').rename([PCName]).set('PC',PCName)
    principalComponentsImages = loadingsImageCollection.map(finalizePCImages)

    # Choose how many principal components are of interest in this analysis based on amount of
    # variance explained
    numberOfComponents = sum(i < propOfVariance for i in cumulativeVariance)+1
    print('Number of Principal Components being used:',numberOfComponents)

    # Compute the combinations of the principal components being used to compute the 2-D convex hulls
    tupleCombinations = list(combinations(list(pcNames[0:numberOfComponents]),2))
    print('Number of Combinations being used:',len(tupleCombinations))

    # Generate convex hulls for an example of the principal components of interest
    cHullCoordsList = list()
    for c in tupleCombinations:
        firstPC = c[0]
        secondPC = c[1]
        outputCHull = ConvexHull(transformedData[[firstPC,secondPC]])
        listOfCoordinates = transformedData.loc[outputCHull.vertices][[firstPC,secondPC]].values.tolist()
        flattenedList = [val for sublist in listOfCoordinates for val in sublist]
        cHullCoordsList.append(flattenedList)

    # Reformat the image collection to an image with band names that can be selected programmatically
    pcImage = principalComponentsImages.toBands().rename(pcNames)

    # Generate an image collection with each PC selected with it's matching PC
    listOfPCs = ee.List(tupleCombinations)
    listOfCHullCoords = ee.List(cHullCoordsList)
    zippedListPCsAndCHulls = listOfPCs.zip(listOfCHullCoords)

    def makeToClassifyImages(zippedListPCsAndCHulls):
        imageToClassify = pcImage.select(ee.List(zippedListPCsAndCHulls).get(0)).set('CHullCoords',ee.List(zippedListPCsAndCHulls).get(1))
        classifiedImage = imageToClassify.rename('u','v').classify(ee.Classifier.spectralRegion([imageToClassify.get('CHullCoords')]))
        return classifiedImage

    classifedImages = ee.ImageCollection(zippedListPCsAndCHulls.map(makeToClassifyImages))
    finalImageToExport = classifedImages.sum().divide(ee.Image.constant(len(tupleCombinations)))

    return finalImageToExport

# PCA interpolation-extrapolation image
#PCA_int_ext = assessExtrapolation(preppedCollection[covariateList_PCA ], propOfVariance).rename('PCA_pct_int_ext')
PCA_int_ext = assessExtrapolation(preppedCollection[covariateList_PCA], propOfVariance, compositeOfInterest).rename('PCA_pct_int_ext')

underExploredMaps = ee.Image.cat(
    #univariate_int_ext_image.rename('univariate_pct_int_ext'),
    PCA_int_ext.rename('PCA_pct_int_ext'))

IntExtclassifiedImageExport = ee.batch.Export.image.toAsset(
     image = underExploredMaps.toFloat(),
     description = classProperty+'_IntExt',
     assetId = 'users/'+usernameFolderString+'/'+projectFolder+'/'+classProperty+'_IntExt',
     crs = 'EPSG:4326',
     crsTransform = '[0.08333333333333333,0,-180,0,-0.08333333333333333,90]',
     region = exportingGeometry,
     maxPixels = int(1e13),
     pyramidingPolicy = {".default": pyramidingPolicy}
 )
IntExtclassifiedImageExport.start()
print("Moving on...")


In [ ]:
# Final image export


print("export stated")
# Construct final image to export
if log_transform_classProperty == True:
    finalImageToExport = ee.Image.cat(
    classifiedImage.
    
    select(0).exp().subtract(1).rename(classProperty+'_Ensemble_mean'),
    meanImage.exp().subtract(1).rename(classProperty+'_Bootstrapped_mean'),
    upperLowerCIImage.select(0).exp().subtract(1).rename(classProperty+'_Bootstrapped_lower'),
    upperLowerCIImage.select(1).exp().subtract(1).rename(classProperty+'_Bootstrapped_upper'),
    stdDevImage.exp().subtract(1).rename(classProperty+'_Bootstrapped_stdDev'),
    coefOfVarImage.exp().subtract(1).rename(classProperty+'_Bootstrapped_coefOfVar'))#,
   ## PCA_int_ext.rename('PCA_pct_int_ext')
else:
    finalImageToExport = ee.Image.cat(
    classifiedImage.select(0).rename(classProperty+'_Ensemble_mean'),
    meanImage.rename(classProperty+'_Bootstrapped_mean'),
    upperLowerCIImage.select(0).rename(classProperty+'_Bootstrapped_lower'),
    upperLowerCIImage.select(1).rename(classProperty+'_Bootstrapped_upper'),
    stdDevImage.rename(classProperty+'_Bootstrapped_stdDev'),
    coefOfVarImage.exp().subtract(1).rename(classProperty+'_Bootstrapped_coefOfVar'))#,
   ## PCA_int_ext.rename('PCA_pct_int_ext')
    
FinalImageExport = ee.batch.Export.image.toAsset(
    image = finalImageToExport.toFloat(),
    description = classProperty+'_Bootstrapped_MultibandImage',
    assetId = 'users/'+usernameFolderString+'/'+projectFolder+'/'+classProperty+'_Classified_MultibandImage_Spring_Grass_Plant',
    crs = 'EPSG:4326',
    crsTransform = '[0.008333333333333333,0,-180,0,-0.008333333333333333,90]',
    region = exportingGeometry,
    maxPixels = int(1e13),
    pyramidingPolicy = {".default": pyramidingPolicy}
)
FinalImageExport.start()

print('Map exports started! Moving on...')